# Generación de SBOMs

## Objetivo

Este notebook explica el proceso para generar **Software Bill of Materials (SBOMs)** para un conjunto de repositorios de código y cómo analizar los resultados de manera centralizada. Utilizaremos la herramienta **Syft** para la generación y la librería **Pandas** en Python para el análisis.

El flujo de trabajo completo está diseñado para ser reproducible y se basa en las siguientes etapas:

1.  **Descubrimiento de Repositorios**: Identificar los proyectos a los que se les generará un SBOM.
2.  **Generación de SBOMs**: Ejecutar Syft para analizar cada repositorio y crear un SBOM en formato JSON.
3.  **Análisis de Resultados**: Cargar todos los SBOMs generados en un DataFrame de Pandas para su inspección y análisis.


## Estructura de Directorios

El proyecto sigue una estructura organizada para separar los datos de entrada, los scripts y los resultados:

```
ciberseguridad_2026/
├── data/
│   ├── repos/      # Directorio para los repositorios a analizar
│   └── results/    # Directorio para los SBOMs generados en JSON
├── nbs/            # Notebooks de Jupyter
└── scripts/        # Scripts de automatización
    └── generate_sboms.py
```

## Paso 1: Generación de SBOMs

El primer paso es ejecutar el script `generate_sboms.py`, que se encarga de orquestar la generación de los SBOMs.

Este script realiza las siguientes acciones:

1.  **Busca repositorios**: Escanea el directorio `data/repos` en busca de subdirectorios que contengan código fuente.
2.  **Ejecuta Syft**: Para cada repositorio encontrado, invoca a `syft` para analizar su contenido.
3.  **Guarda los resultados**: El SBOM generado para cada repositorio se guarda como un archivo JSON en el directorio `data/results`.

In [1]:
#| eval: false
!python ../../scripts/generate_sboms.py

INFO | Usando Syft CLI: /usr/local/bin/syft
INFO | [1/9] Procesando repositorio data/repos/annotated-doc
INFO | SBOM guardado en data/results/annotated-doc-sbom.json
INFO | [2/9] Procesando repositorio data/repos/asyncer
INFO | SBOM guardado en data/results/asyncer-sbom.json
INFO | [3/9] Procesando repositorio data/repos/fastapi
INFO | SBOM guardado en data/results/fastapi-sbom.json
INFO | [4/9] Procesando repositorio data/repos/fastapi-cli
INFO | SBOM guardado en data/results/fastapi-cli-sbom.json
INFO | [5/9] Procesando repositorio data/repos/fastapi-new
INFO | SBOM guardado en data/results/fastapi-new-sbom.json
INFO | [6/9] Procesando repositorio data/repos/fastapi-vscode
INFO | SBOM guardado en data/results/fastapi-vscode-sbom.json
INFO | [7/9] Procesando repositorio data/repos/full-stack-fastapi-template
INFO | SBOM guardado en data/results/full-stack-fastapi-template-sbom.json
INFO | [8/9] Procesando repositorio data/repos/sqlmodel
INFO | SBOM guardado en data/results/sqlmodel-sb

## Paso 2: Análisis de los SBOMs con Pandas

Una vez que los SBOMs han sido generados, podemos cargarlos en un DataFrame de Pandas para analizarlos. Esto nos permite tener una visión consolidada de todas las dependencias y artefactos encontrados en los diferentes repositorios.

El siguiente código se encarga de:

1.  **Localizar los archivos JSON**: Busca todos los archivos `.json` en el directorio `data/results`.
2.  **Cargar y normalizar los datos**: Para cada archivo JSON, lo carga y utiliza `pd.json_normalize` para aplanar la estructura anidada de los artefactos.
3.  **Añadir información del repositorio**: Agrega una columna `repo` para identificar a qué repositorio pertenece cada artefacto.
4.  **Concatenar los resultados**: Une todos los DataFrames individuales en uno solo.

In [3]:
import pandas as pd
from pathlib import Path
import json

path = Path("../../data/results")

df_all = pd.concat(
    [
        pd.json_normalize(json.load(open(file))[
                          "artifacts"]).assign(repo=file.stem)
        for file in path.glob("*-sbom.json")
    ],
    ignore_index=True
)

df_all.head()

,id,name,version,type,foundBy,locations,licenses,language,cpes,purl,metadataType,metadata.value,metadata.comment,metadata.index,metadata.dependencies,metadata.extras,repo
0,e07ae9d290a8143c,CodSpeedHQ/action,v4.12.1,github-action,github-actions-usage-cataloger,"[{'path': '/.github/workflows/test.yml', 'acce...",[],,[{'cpe': 'cpe:2.3:a:CodSpeedHQ\/action:CodSpee...,pkg:github/CodSpeedHQ/action@v4.12.1,github-actions-use-statement,CodSpeedHQ/action@1c8ae4843586d3ba879736b7f6b7...,v4.12.1,NaN,NaN,NaN,fastapi-sbom
1,137e914427cf2343,a2wsgi,1.10.10,python,python-package-cataloger,"[{'path': '/uv.lock', 'accessPath': '/uv.lock'...",[],python,[{'cpe': 'cpe:2.3:a:python-a2wsgi:python-a2wsg...,pkg:pypi/a2wsgi@1.10.10,python-uv-lock-entry,NaN,NaN,https://pypi.org/simple,"[{'name': 'typing-extensions', 'optional': Fal...",NaN,fastapi-sbom
2,33b22f26dfca139e,actions/add-to-project,v1.0.2,github-action,github-actions-usage-cataloger,[{'path': '/.github/workflows/add-to-project.y...,[],,[{'cpe': 'cpe:2.3:a:actions\/add-to-project:ac...,pkg:github/actions/add-to-project@v1.0.2,github-actions-use-statement,actions/add-to-project@244f685bbc3b7adfa8466e0...,v1.0.2,NaN,NaN,NaN,fastapi-sbom
3,423ad96cdd3a4dab,actions/cache,v5.0.5,github-action,github-actions-usage-cataloger,"[{'path': '/.github/workflows/build-docs.yml',...",[],,[{'cpe': 'cpe:2.3:a:actions\/cache:actions\/ca...,pkg:github/actions/cache@v5.0.5,github-actions-use-statement,actions/cache@27d5ce7f107fe9357f9df03efb73ab90...,v5.0.5,NaN,NaN,NaN,fastapi-sbom
4,2c646db9addcc600,actions/checkout,v6.0.2,github-action,github-actions-usage-cataloger,"[{'path': '/.github/workflows/build-docs.yml',...",[],,[{'cpe': 'cpe:2.3:a:actions\/checkout:actions\...,pkg:github/actions/checkout@v6.0.2,github-actions-use-statement,actions/checkout@de0fac2e4500dabe0009e67214ff5...,v6.0.2,NaN,NaN,NaN,fastapi-sbom


## Exploración de los Resultados

### Cantidad de artefactos según su tipo

In [10]:
df_all['type'].value_counts()

type
python           735
github-action    331
Name: count, dtype: int64

#### Lenguaje principales de los artefactos

In [11]:
df_all['language'].value_counts()

language
python    735
          331
Name: count, dtype: int64

Los 331 no presentan lenguaje porque seguramente son de tipo Gibhub actions (tal como muestra la _query_ anterior)

### Se busca las relaciones entre dependencias más frecuentes

In [9]:

artifact_relationships_all = pd.concat(
    [
        pd.json_normalize(json.load(open(file))[
                          "artifactRelationships"]).assign(repo=file.stem)
        for file in path.glob("*-sbom.json")
    ],
    ignore_index=True
)

artifact_relationships_all.value_counts("type")

type
dependency-of    1230
evident-by       1066
contains         1066
Name: count, dtype: int64

### Artefactos por cada repositorio

In [4]:
df_all['repo'].value_counts()

repo
fastapi-sbom                        316
typer-sbom                          135
sqlmodel-sbom                       135
asyncer-sbom                        133
full-stack-fastapi-template-sbom    118
fastapi-cli-sbom                     83
annotated-doc-sbom                   70
fastapi-new-sbom                     64
fastapi-vscode-sbom                  12
Name: count, dtype: int64

Se muestra que el proyecto principal (_FastAPI_) es el que presenta mayor cantidad de artefactos.

### Licencias encontradas en los artefactos

In [5]:
df_all['licenses'].explode().value_counts().head(10)

Series([], Name: count, dtype: int64)

Se puede apreciar que no se encontraron licencias en ninguno de los artefactos.

### Paquetes compartidos entre repos

In [ ]:
common = (
    df_all.dropna(subset=["purl"])
    .groupby("purl")["repo"].nunique()
    .sort_values(ascending=False)
)
common.head(20)

purl
pkg:github/tiangolo/latest-changes@0.4.1                                      9
pkg:github/actions/labeler@v6.0.1                                             9
pkg:github/actions/checkout@v6.0.2                                            9
pkg:github/actions/download-artifact@v8.0.1                                   9
pkg:github/agilepathway/label-checker@v1.6.65                                 9
pkg:pypi/shellingham@1.5.4                                                    8
pkg:github/pre-commit-ci/lite-action@v1.1.0                                   8
pkg:pypi/mdurl@0.1.2                                                          8
pkg:github/mxschmitt/action-tmate@c0afd6f790e3a5564914980036ebf83216678101    8
pkg:pypi/idna@3.11                                                            8
pkg:pypi/httpcore@1.0.9                                                       8
pkg:pypi/httpx@0.28.1                                                         8
pkg:pypi/mypy-extensions@1.1.0     

## Cómo Añadir Nuevos Repositorios al Análisis

El proyecto está diseñado para que sea sencillo añadir nuevos repositorios al proceso de generación de SBOMs. El sistema utiliza **Git submodules** para gestionar los repositorios externos.

#### 1. Edita el archivo `data/repos.json`

Abre `data/repos.json` y agrega nuevos repositorios con esta estructura:

```json
{
  "url": "URL_DEL_REPOSITORIO_EN_GITHUB.git",
  "path": "data/repos/NOMBRE_DEL_REPOSITORIO"
}
```

- `url`: La URL `.git` del repositorio
- `path`: Ruta local donde se clonará (usa `data/repos/{nombre}`)

#### 2. Ejecuta en la terminal (esto clona los repositorios y los agrega como submódulos):

```bash
uv run scripts/add_submodules.py && git submodule update --init --recursive
```

#### 3. Ejecuta la generación de SBOMs nuevamente:

```bash
uv run scripts/generate_sboms.py
```

O desde el notebook:
- Reinicia el kernel
- Ejecuta la celda con `!python ../../scripts/generate_sboms.py`
